# Demo ADASYN 

## Import và cấu hình pipeline

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import (
    RANDOM_STATE,
    TEST_SIZE,
    build_leakage_checklist,
    make_features_target,
    preprocess_train_test,
    split_data,
)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"TEST_SIZE = {TEST_SIZE}")
print(f"RANDOM_STATE = {RANDOM_STATE}")

## Quang - Load data và kiểm tra dữ liệu đầu vào

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "uci_wdbc" / "wdbc.data"

WDBC_COLUMNS = [
    "id",
    "diagnosis",
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave_points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",
    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave_points_se",
    "symmetry_se",
    "fractal_dimension_se",
    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave_points_worst",
    "symmetry_worst",
    "fractal_dimension_worst",
]

ID_COLUMN = "id"
TARGET_COLUMN = "diagnosis"
FEATURE_COLUMNS = [column for column in WDBC_COLUMNS if column not in [ID_COLUMN, TARGET_COLUMN]]

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset file not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, header=None, names=WDBC_COLUMNS)
df[FEATURE_COLUMNS] = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")

missing_values = df.isna().sum()
numeric_feature_count = df[FEATURE_COLUMNS].select_dtypes(include="number").shape[1]

input_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "feature columns",
            "numeric feature columns",
            "target column",
            "total missing values",
        ],
        "value": [
            df.shape[0],
            df.shape[1],
            len(FEATURE_COLUMNS),
            numeric_feature_count,
            TARGET_COLUMN,
            int(missing_values.sum()),
        ],
    }
)

target_preview = df[TARGET_COLUMN].value_counts().rename_axis("class").reset_index(name="count")

print(f"Loaded dataset from: {DATA_PATH}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Target column: {TARGET_COLUMN}")
print(f"Feature columns: {len(FEATURE_COLUMNS)}")
print(f"Total missing values: {int(missing_values.sum())}")

display(input_summary)
display(target_preview)
display(df.head())

## Quang - EDA và class distribution

In [ ]:
required_names = ["df", "TARGET_COLUMN"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(f"Missing variables from load-data cell: {missing_names}")

CLASS_LABELS = {"M": "Malignant", "B": "Benign"}

class_distribution = (
    df[TARGET_COLUMN]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="count")
)
class_distribution["label"] = class_distribution["class"].map(CLASS_LABELS)
class_distribution["percentage"] = (class_distribution["count"] / len(df) * 100).round(2)

majority_index = class_distribution["count"].idxmax()
minority_index = class_distribution["count"].idxmin()
majority_class = class_distribution.loc[majority_index, "class"]
minority_class = class_distribution.loc[minority_index, "class"]
majority_count = int(class_distribution.loc[majority_index, "count"])
minority_count = int(class_distribution.loc[minority_index, "count"])
class_ratio = round(majority_count / minority_count, 2)

class_distribution["class_type"] = class_distribution["class"].map(
    {majority_class: "Majority", minority_class: "Minority"}
)

print(f"Majority class: {majority_class} ({majority_count} samples)")
print(f"Minority class: {minority_class} ({minority_count} samples)")
print(f"Majority/minority ratio: {class_ratio}:1")

display(class_distribution)

import matplotlib.pyplot as plt

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
class_distribution_path = RESULTS_DIR / "class_distribution.png"

plot_labels = class_distribution["class"] + " - " + class_distribution["label"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(plot_labels, class_distribution["count"], color=["#4C78A8", "#F58518"])
ax.set_title("WDBC Class Distribution")
ax.set_xlabel("Diagnosis class")
ax.set_ylabel("Sample count")
ax.set_ylim(0, class_distribution["count"].max() * 1.18)

for bar, percentage in zip(bars, class_distribution["percentage"]):
    height = bar.get_height()
    ax.annotate(
        f"{int(height)}\n{percentage:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
    )

fig.tight_layout()
fig.savefig(class_distribution_path, dpi=150, bbox_inches="tight")
if plt.get_backend().lower() == "agg":
    plt.close(fig)
else:
    plt.show()

print(f"Saved class distribution chart to: {class_distribution_path}")

## Phúc - Preprocessing và train/test split chuẩn

In [ ]:
# Chạy cell này sau khi phần Quang đã tạo đủ 3 biến:
# - df: DataFrame dữ liệu đã load
# - FEATURE_COLUMNS: danh sách cột feature
# - TARGET_COLUMN: tên cột target

required_names = ["df", "FEATURE_COLUMNS", "TARGET_COLUMN"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(f"Thiếu biến đầu vào từ phần Quang: {missing_names}")

X, y = make_features_target(df, FEATURE_COLUMNS, TARGET_COLUMN)
X_train, X_test, y_train, y_test = split_data(X, y)
X_train_processed, X_test_processed, preprocessor = preprocess_train_test(X_train, X_test)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"X_train_processed: {X_train_processed.shape}")
print(f"X_test_processed: {X_test_processed.shape}")

## Quân - Baseline, Random Oversampling, Random Undersampling

In [ ]:
#

## Phước - SMOTE, ADASYN và thử `k_neighbors`

In [ ]:
#

## Quyên - Metrics, confusion matrix và biểu đồ so sánh

In [ ]:
#

## Phúc - Checklist chống data leakage

In [ ]:
required_names = ["X_train", "X_test", "X_train_processed", "X_test_processed"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(f"Thiếu biến để kiểm tra leakage: {missing_names}")

leakage_checks = build_leakage_checklist(
    X_train,
    X_test,
    X_train_processed,
    X_test_processed,
)

display(leakage_checks)
assert leakage_checks["passed"].all(), "Có checklist data leakage chưa đạt."
print("Checklist data leakage đạt cho phần pipeline của Phúc.")

## Phúc - README, requirements và bàn giao khung notebook